# Module 09B - Pretraining

Use this notebook after Module 09's `TransformerLM` is working. The goal is to make the language-model training data concrete: one token stream becomes train/validation streams, `(B, T)` shifted batches, `(B, T, V)` logits, and one scalar cross-entropy loss.

1. Read the lesson page (`docs/modules/09b-pretraining.md`).
2. Open this notebook with `.venv/bin/python scripts/open_notebook.py 09b`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [1]:
from __future__ import annotations

import math
import subprocess
import sys
from pathlib import Path

import torch

import g2c
from g2c.pretraining import get_lm_batch, lm_cross_entropy, split_token_stream
from g2c.transformer import TransformerLM

_ = torch.manual_seed(0)
repo_root = Path(g2c.__file__).resolve().parents[1]
print("repo root:", repo_root)


repo root: /Users/colkitt/sith/toys/courses/g2c


## Before the Notebook

`split_token_stream` and `get_lm_batch` are implemented for you. Implement `lm_cross_entropy` before working through the later cells.

In [2]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_pretraining_setup.py -x"
"Question: Which setup test is the next one failing, and what shape contract does it point at?"
"Answer: "


'Answer: '

In [3]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_pretraining_setup.py"],
    cwd=repo_root,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 09B pretraining setup tests are not passing yet."
print("Module 09B pretraining setup tests passed.")


...............                                                          [100%]
15 passed in 0.58s

Module 09B pretraining setup tests passed.


## Exercise 1 - Shift a Toy Stream

Start with a sequence where the answer is visible by inspection.

In [4]:
ids = torch.tensor([10, 11, 12, 13, 14, 15])
start = 1
T = 3
x = ids[start : start + T]
y = ids[start + 1 : start + T + 1]
print("ids:", ids.tolist())
print("x:", x.tolist())
print("y:", y.tolist())


ids: [10, 11, 12, 13, 14, 15]
x: [11, 12, 13]
y: [12, 13, 14]


In [5]:
"Question: Why is y[t] the training target for x[t]?"
"Answer: "


'Answer: '

## Exercise 2 - Split a Token Stream

A language-model split preserves order inside each split. Randomness comes from sampling windows later, not from shuffling individual tokens.

In [6]:
toy_stream = torch.arange(30)
train_ids, val_ids = split_token_stream(toy_stream, train_fraction=0.8)
print("train:", train_ids.tolist())
print("val:", val_ids.tolist())
print("train tokens:", len(train_ids))
print("val tokens:", len(val_ids))


train: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
val: [24, 25, 26, 27, 28, 29]
train tokens: 24
val tokens: 6


In [7]:
"Question: Why would shuffling individual token IDs break next-token prediction?"
"Answer: "


'Answer: '

## Exercise 3 - Sample Multi-Position Batches

`get_lm_batch` samples random contiguous windows. With `torch.arange`, every target should equal the matching input plus one.

In [8]:
ids = torch.arange(30)
xb, yb = get_lm_batch(
    ids,
    batch_size=4,
    context_length=6,
    generator=torch.Generator().manual_seed(0),
)
print("x shape:", tuple(xb.shape))
print("y shape:", tuple(yb.shape))
print("x:\n", xb)
print("y:\n", yb)
print("y == x + 1:", torch.equal(yb, xb + 1))
assert xb.shape == (4, 6)
assert yb.shape == (4, 6)
assert torch.equal(yb, xb + 1)


x shape: (4, 6)
y shape: (4, 6)
x:
 tensor([[20, 21, 22, 23, 24, 25],
        [15, 16, 17, 18, 19, 20],
        [ 5,  6,  7,  8,  9, 10],
        [ 0,  1,  2,  3,  4,  5]])
y:
 tensor([[21, 22, 23, 24, 25, 26],
        [16, 17, 18, 19, 20, 21],
        [ 6,  7,  8,  9, 10, 11],
        [ 1,  2,  3,  4,  5,  6]])
y == x + 1: True


In [9]:
"Question: Why does this one (B, T) batch contain B*T classification examples?"
"Answer: "


'Answer: '

## Exercise 4 - Language-Model Cross-Entropy

The loss is ordinary cross-entropy after flattening positions.

In [10]:
B, T, V = 2, 4, 7
logits = torch.zeros(B, T, V)
targets = torch.randint(0, V, (B, T), generator=torch.Generator().manual_seed(0))
loss = lm_cross_entropy(logits, targets)
print("logits shape:", tuple(logits.shape))
print("targets shape:", tuple(targets.shape))
print("flat logits shape:", (B * T, V))
print("flat targets shape:", (B * T,))
print("loss:", float(loss))
print("log(V):", math.log(V))
assert abs(loss.item() - math.log(V)) < 1e-5


logits shape: (2, 4, 7)
targets shape: (2, 4)
flat logits shape: (8, 7)
flat targets shape: (8,)
loss: 1.945910096168518
log(V): 1.9459101490553132


In [11]:
"Question: What reshape turns logits from (B, T, V) into the shape CrossEntropyLoss expects?"
"Answer: "


'Answer: '

## Exercise 5 - Baselines for Different Vocabulary Sizes

The uniform-loss baseline rises with vocabulary size because uniform guessing has more classes to choose from.

In [12]:
for V in [256, 512, 1024]:
    loss = math.log(V)
    print(f"V={V:4d}  log(V)={loss:.3f}  perplexity={math.exp(loss):.0f}")


V= 256  log(V)=5.545  perplexity=256
V= 512  log(V)=6.238  perplexity=512
V=1024  log(V)=6.931  perplexity=1024


In [13]:
"Question: Why can a larger vocabulary start with larger loss but still be useful?"
"Answer: "


'Answer: '

## Exercise 6 - Random Transformer Sanity Check

A random model should usually start near `log(V)`. This does not mean it is good; it means the objective is wired plausibly before training.

In [14]:
torch.manual_seed(0)
vocab_size = 64
model = TransformerLM(
    vocab_size=vocab_size,
    embedding_dim=16,
    num_layers=1,
    num_heads=2,
    max_seq_len=8,
)
ids = torch.randint(0, vocab_size, (500,), generator=torch.Generator().manual_seed(1))
xb, yb = get_lm_batch(ids, batch_size=8, context_length=8, generator=torch.Generator().manual_seed(2))
logits = model(xb)
loss = lm_cross_entropy(logits, yb)
print("logits shape:", tuple(logits.shape))
print("loss:", float(loss))
print("log(V):", math.log(vocab_size))


logits shape: (8, 8, 64)
loss: 4.361391544342041
log(V): 4.1588830833596715


/var/folders/sd/yrr853rn73qdgdgdmps82gfh0000gn/T/ipykernel_36382/2552432934.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print("loss:", float(loss))


In [15]:
"Question: If the random-model loss is far below log(V), what are two possible explanations?"
"Answer: "


'Answer: '

When complete, ask a coding agent to grade your Module 09B notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.